# Comprehensive Vision Dataset Analysis & Visualization Framework
This notebook performs a deep-dive EDA on the multi-class plant disease image dataset. It automatically analyzes class balance, image dimensions, pixel color intensities, and provides previews of training augmentations.

In [ ]:
import os
import random
import glob
import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import torch
import torchvision.transforms.v2 as v2

# Apply clean global styling
plt.style.use('ggplot')
sns.set_context('notebook')

DATASET_DIR = '../DATASET/Original'

## 1. Class Distribution & Balance
Understanding the volume of images per class is critical. Significant imbalances can bias the neural network. We visualize the absolute counts and the broader 'Healthy vs Diseased' proportion.

In [ ]:
# Count images per class
class_counts = {}
for cls_dir in os.listdir(DATASET_DIR):
    cls_path = os.path.join(DATASET_DIR, cls_dir)
    if os.path.isdir(cls_path):
        class_counts[cls_dir] = len(os.listdir(cls_path))

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Count'])
df_counts = df_counts.sort_values('Count', ascending=False).reset_index(drop=True)
df_counts['Type'] = df_counts['Class'].apply(lambda x: 'Healthy' if 'healthy' in x.lower() else 'Diseased')

fig, ax = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [2, 1]})

# Bar Chart
sns.barplot(data=df_counts, y='Class', x='Count', hue='Type', dodge=False, ax=ax[0])
ax[0].set_title('Absolute Image Count per Class', fontsize=16, fontweight='bold')
ax[0].set_xlabel('Number of Images', fontsize=12)
ax[0].set_ylabel('Class Label', fontsize=12)

# Pie Chart
type_counts = df_counts.groupby('Type')['Count'].sum()
ax[1].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140, 
          colors=['#ff9999', '#66b3ff'], explode=(0.05, 0), shadow=True, textprops={'fontsize': 14})
ax[1].set_title('Healthy vs Diseased Distribution', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

### Hierarchical Treemap
We can extract the Plant Species from the class name (usually denoted before `___`) to visualize the broad dataset hierarchy.

In [ ]:
# Extract Plant Name
df_counts['Plant'] = df_counts['Class'].apply(lambda x: x.split('___')[0] if '___' in x else 'Unknown')
df_counts['Condition'] = df_counts['Class'].apply(lambda x: x.split('___')[1] if '___' in x else x)

# Generate Treemap
fig = px.treemap(df_counts, path=['Type', 'Plant', 'Condition'], values='Count',
                 color='Count', hover_data=['Class'],
                 color_continuous_scale='RdYlGn',
                 title='Hierarchical Distribution of the Dataset')
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25), height=600)
fig.show()

## 2. Image Metadata & Geometry
Images from the web or public datasets often vary drastically in resolution. Plotting the widths vs heights and observing aspect ratios helps us decide the optimal standardized `IMG_SIZE` for our CNN.

In [ ]:
import multiprocessing
from collections import Counter

print('Scanning images to extract geometrical metadata... (This might take a moment)\n')

widths, heights, aspect_ratios = [], [], []

# To keep it fast, we only open image headers (PIL does this automatically when getting .size)
all_images = glob.glob(os.path.join(DATASET_DIR, '*/*.*'))

for img_path in all_images:
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(round(w / h, 2))
    except Exception:
        pass

print(f"Extracted metadata from {len(widths)} images.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of Aspect Ratios
sns.histplot(aspect_ratios, bins=30, kde=True, ax=ax[0], color='purple')
ax[0].set_title('Distribution of Image Aspect Ratios (Width/Height)', fontsize=14)
ax[0].set_xlabel('Aspect Ratio', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].axvline(np.mean(aspect_ratios), color='red', linestyle='--', label=f'Mean: {np.mean(aspect_ratios):.2f}')
ax[0].legend()

# Scatter plot width vs height
sns.scatterplot(x=widths, y=heights, alpha=0.3, ax=ax[1], color='teal')
ax[1].set_title('Image Resolution (Width vs Height)', fontsize=14)
ax[1].set_xlabel('Width (pixels)', fontsize=12)
ax[1].set_ylabel('Height (pixels)', fontsize=12)

plt.tight_layout()
plt.show()

## 3. Color & Intensity Analysis
To understand the pixel-level distributions, we plot RGB histograms and general average brightness using Kernel Density Estimates (KDE).
> **Note:** To ensure the notebook remains responsive, we aggressively sample `N=50` images per class for this pixel-level analysis.

In [ ]:
SAMPLES_PER_CLASS = 50
sampled_pixels = []
mean_brightness = []

print("Sampling images for Color & Intensity profiling...")
for cls_dir in os.listdir(DATASET_DIR):
    cls_path = os.path.join(DATASET_DIR, cls_dir)
    if os.path.isdir(cls_path):
        images_in_cls = os.listdir(cls_path)
        sample_size = min(SAMPLES_PER_CLASS, len(images_in_cls))
        sampled = random.sample(images_in_cls, sample_size)
        
        for img_name in sampled:
            img_path = os.path.join(cls_path, img_name)
            try:
                with Image.open(img_path).convert('RGB') as img:
                    # Resize to tiny thumbnail to quickly capture overall pixel distribution
                    img_thumb = img.resize((64, 64))
                    arr = np.array(img_thumb) / 255.0
                    sampled_pixels.append(arr)
                    
                    # Calculate relative luminance (Rec. 601 weighting)
                    lum = 0.299 * arr[:,:,0] + 0.587 * arr[:,:,1] + 0.114 * arr[:,:,2]
                    mean_brightness.append(lum.mean())
            except Exception:
                pass

sampled_pixels = np.array(sampled_pixels)
print(f"Compiled pixel distributions from {len(sampled_pixels)} sampled images.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# RGB Histograms
# Flatten pixel arrays per channel
pixels_r = sampled_pixels[:,:,:,0].ravel()
pixels_g = sampled_pixels[:,:,:,1].ravel()
pixels_b = sampled_pixels[:,:,:,2].ravel()

sns.kdeplot(pixels_r, color='red', ax=ax[0], label='Red', fill=True, alpha=0.1)
sns.kdeplot(pixels_g, color='green', ax=ax[0], label='Green', fill=True, alpha=0.1)
sns.kdeplot(pixels_b, color='blue', ax=ax[0], label='Blue', fill=True, alpha=0.1)
ax[0].set_title('RGB Global Intensity Distribution (Normalized)', fontsize=14)
ax[0].set_xlabel('Pixel Intensity [0, 1]')
ax[0].set_ylabel('Density')
ax[0].legend()
ax[0].set_xlim([0, 1])

# Brightness KDE
sns.kdeplot(mean_brightness, ax=ax[1], color='darkorange', fill=True, linewidth=2)
ax[1].set_title('Average Brightness (Luminance) KDE', fontsize=14)
ax[1].set_xlabel('Mean Brightness [0: Dark, 1: Bright]')
ax[1].set_ylabel('Density')
ax[1].axvline(np.mean(mean_brightness), color='black', linestyle='--', label=f'Overall Mean: {np.mean(mean_brightness):.2f}')
ax[1].legend()
ax[1].set_xlim([0, 1])

plt.tight_layout()
plt.show()

## 4. Augmentation Simulator
Data augmentation creates distinct combinations of inputs dynamically during CNN training to severely limit overfitting. Use this simulator grid to visually verify your transform parameters.

In [ ]:
def preview_augmentations(img_path):
    img = Image.open(img_path).convert('RGB')
    
    # Define individual augmentations
    rot = v2.RandomRotation(degrees=(30, 30))(img)
    flip = v2.RandomHorizontalFlip(p=1.0)(img)
    color = v2.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.3)(img)
    zoom = v2.RandomResizedCrop(size=img.size, scale=(0.5, 0.5))(img)
    
    imgs = [img, rot, color, flip, zoom]
    titles = ['Raw Image', 'Rotated (30°)', 'Color Jittered', 'Horizontally Flipped', 'Zoomed (Resized Crop)']
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i])
        ax.set_title(titles[i], fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Pick random image from dataset
random_cls = random.choice(os.listdir(DATASET_DIR))
random_img = random.choice(os.listdir(os.path.join(DATASET_DIR, random_cls)))
random_path = os.path.join(DATASET_DIR, random_cls, random_img)

preview_augmentations(random_path)